<a href="https://colab.research.google.com/github/Suprithaaaa01/eeg-cognitive-state-classifier/blob/main/eegnet_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install mne mne-bids -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 76.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 189.8/189.8 kB 19.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.


In [2]:
import mne
print(mne.__version__)

1.12.1


In [3]:
from mne.datasets import eegbci
import mne

mne.set_log_level('WARNING')

subjects = list(range(1, 21))
runs = [4, 8, 12]

print(f"Downloading {len(subjects)} subjects...")
for subject in subjects:
    eegbci.load_data(subject, runs, path='./data', update_path=True)
    print(f"Subject {subject} done")

print("Download complete!")

Subject 1 done


Subject 2 done


Subject 3 done


Subject 4 done


Subject 5 done


...

Subject 16 done


Subject 17 done


Subject 18 done


Subject 19 done


Subject 20 done
Download complete!


In [4]:
import glob
import numpy as np

def load_subject_epochs(subject_num):
    """Load and epoch all 3 runs for one subject, return combined Epochs object."""
    pattern = f'data/MNE-eegbci-data/files/eegmmidb/1.0.0/S{subject_num:03d}/S{subject_num:03d}R*.edf'
    files = sorted(glob.glob(pattern))
    subject_epochs = []
    for filepath in files:
        raw = mne.io.read_raw_edf(filepath, preload=True)
        raw.filter(7., 30., fir_design='firwin', skip_by_annotation='edge')
        events, event_id = mne.events_from_annotations(raw)
        event_id_2class = {k: v for k, v in event_id.items() if k in ('T1', 'T2')}
        epochs = mne.Epochs(raw, events, event_id_2class, tmin=1.0, tmax=3.0,
                             baseline=None, preload=True)
        subject_epochs.append(epochs)
    return mne.concatenate_epochs(subject_epochs)

# Same subject-wise split as before: train 1-16, test 17-20 (unseen)
train_subjects = list(range(1, 17))
test_subjects = list(range(17, 21))

print("Loading training subjects...")
train_epochs_list = [load_subject_epochs(s) for s in train_subjects]
train_epochs = mne.concatenate_epochs(train_epochs_list)

print("Loading test subjects...")
test_epochs_list = [load_subject_epochs(s) for s in test_subjects]
test_epochs = mne.concatenate_epochs(test_epochs_list)

X_train = train_epochs.get_data().astype(np.float32)
y_train = train_epochs.events[:, -1] - 2   # convert labels 2,3 -> 0,1
X_test = test_epochs.get_data().astype(np.float32)
y_test = test_epochs.events[:, -1] - 2

print(f"\nTrain shape: {X_train.shape}, label dist: {np.bincount(y_train)}")
print(f"Test shape: {X_test.shape}, label dist: {np.bincount(y_test)}")

Loading training subjects...


/tmp/ipykernel_608/3716197497.py:17: RuntimeWarning: Concatenation of Annotations within Epochs is not supported yet. All annotations will be dropped.
  return mne.concatenate_epochs(subject_epochs)
/tmp/ipykernel_608/3716197497.py:17: RuntimeWarning: Concatenation of Annotations within Epochs is not supported yet. All annotations will be dropped.
  return mne.concatenate_epochs(subject_epochs)
/tmp/ipykernel_608/3716197497.py:17: RuntimeWarning: Concatenation of Annotations within Epochs is not supported yet. All annotations will be dropped.
  return mne.concatenate_epochs(subject_epochs)
/tmp/ipykernel_608/3716197497.py:17: RuntimeWarning: Concatenation of Annotations within Epochs is not supported yet. All annotations will be dropped.
  return mne.concatenate_epochs(subject_epochs)
/tmp/ipykernel_608/3716197497.py:17: RuntimeWarning: Concatenation of Annotations within Epochs is not supported yet. All annotations will be dropped.
  return mne.concatenate_epochs(subject_epochs)
/tmp/

Loading test subjects...


/tmp/ipykernel_608/3716197497.py:17: RuntimeWarning: Concatenation of Annotations within Epochs is not supported yet. All annotations will be dropped.
  return mne.concatenate_epochs(subject_epochs)
/tmp/ipykernel_608/3716197497.py:17: RuntimeWarning: Concatenation of Annotations within Epochs is not supported yet. All annotations will be dropped.
  return mne.concatenate_epochs(subject_epochs)
/tmp/ipykernel_608/3716197497.py:17: RuntimeWarning: Concatenation of Annotations within Epochs is not supported yet. All annotations will be dropped.
  return mne.concatenate_epochs(subject_epochs)



Train shape: (720, 64, 321), label dist: [364 356]
Test shape: (180, 64, 321), label dist: [91 89]


/tmp/ipykernel_608/3716197497.py:17: RuntimeWarning: Concatenation of Annotations within Epochs is not supported yet. All annotations will be dropped.
  return mne.concatenate_epochs(subject_epochs)


In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class EEGNet(nn.Module):
    def __init__(self, n_channels=64, n_samples=321, n_classes=2,
                 F1=8, D=2, F2=16, kernel_length=64, dropout=0.5):
        super().__init__()

        # Block 1: temporal convolution -- learns frequency-like filters over time
        self.firstconv = nn.Sequential(
            nn.Conv2d(1, F1, (1, kernel_length), padding=(0, kernel_length // 2), bias=False),
            nn.BatchNorm2d(F1)
        )

        # Depthwise conv: spatial filtering, one filter set per channel group -- this is the CSP-like step
        self.depthwiseConv = nn.Sequential(
            nn.Conv2d(F1, F1 * D, (n_channels, 1), groups=F1, bias=False),
            nn.BatchNorm2d(F1 * D),
            nn.ELU(),
            nn.AvgPool2d((1, 4)),
            nn.Dropout(dropout)
        )

        # Separable conv: combines feature maps efficiently
        self.separableConv = nn.Sequential(
            nn.Conv2d(F1 * D, F2, (1, 16), padding=(0, 8), groups=F1 * D, bias=False),
            nn.Conv2d(F2, F2, 1, bias=False),
            nn.BatchNorm2d(F2),
            nn.ELU(),
            nn.AvgPool2d((1, 8)),
            nn.Dropout(dropout)
        )

        # Figure out flattened size dynamically
        with torch.no_grad():
            dummy = torch.zeros(1, 1, n_channels, n_samples)
            out = self.separableConv(self.depthwiseConv(self.firstconv(dummy)))
            flat_size = out.view(1, -1).shape[1]

        self.classify = nn.Linear(flat_size, n_classes)

    def forward(self, x):
        # x shape: (batch, channels, timepoints) -> add a dim for conv2d: (batch, 1, channels, timepoints)
        x = x.unsqueeze(1)
        x = self.firstconv(x)
        x = self.depthwiseConv(x)
        x = self.separableConv(x)
        x = x.view(x.size(0), -1)
        return self.classify(x)

# Quick sanity check
model = EEGNet(n_channels=64, n_samples=321, n_classes=2)
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

EEGNet(
  (firstconv): Sequential(
    (0): Conv2d(1, 8, kernel_size=(1, 64), stride=(1, 1), padding=(0, 32), bias=False)
    (1): BatchNorm2d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (depthwiseConv): Sequential(
    (0): Conv2d(8, 16, kernel_size=(64, 1), stride=(1, 1), groups=8, bias=False)
    (1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ELU(alpha=1.0)
    (3): AvgPool2d(kernel_size=(1, 4), stride=(1, 4), padding=0)
    (4): Dropout(p=0.5, inplace=False)
  )
  (separableConv): Sequential(
    (0): Conv2d(16, 16, kernel_size=(1, 16), stride=(1, 1), padding=(0, 8), groups=16, bias=False)
    (1): Conv2d(16, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (3): ELU(alpha=1.0)
    (4): AvgPool2d(kernel_size=(1, 8), stride=(1, 8), padding=0)
    (5): Dropout(p=0.5, inplace=False)
  )
  (classify): Linear(in_feature

In [6]:
from torch.utils.data import TensorDataset, DataLoader
import numpy as np

# Standardize features (z-score per channel, fit on train only)
mean = X_train.mean(axis=(0, 2), keepdims=True)
std = X_train.std(axis=(0, 2), keepdims=True) + 1e-8
X_train_norm = (X_train - mean) / std
X_test_norm = (X_test - mean) / std

# Convert to tensors
X_train_t = torch.tensor(X_train_norm, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
X_test_t = torch.tensor(X_test_norm, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.long)

train_ds = TensorDataset(X_train_t, y_train_t)
test_ds = TensorDataset(X_test_t, y_test_t)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)

# Set up device, model, optimizer, loss
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

model = EEGNet(n_channels=64, n_samples=321, n_classes=2).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()

def evaluate(loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            preds = model(xb).argmax(dim=1)
            correct += (preds == yb).sum().item()
            total += yb.size(0)
    return correct / total

# Training loop
n_epochs = 100
for epoch in range(n_epochs):
    model.train()
    total_loss = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        out = model(xb)
        loss = criterion(out, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    if (epoch + 1) % 10 == 0:
        train_acc = evaluate(train_loader)
        test_acc = evaluate(test_loader)
        print(f"Epoch {epoch+1}/{n_epochs} - Loss: {total_loss/len(train_loader):.4f} - Train acc: {train_acc:.3f} - Test acc: {test_acc:.3f}")

print("\nTraining complete!")
final_test_acc = evaluate(test_loader)
print(f"Final test accuracy: {final_test_acc:.3f}")

Using device: cuda
Epoch 10/100 - Loss: 0.6844 - Train acc: 0.667 - Test acc: 0.506
Epoch 20/100 - Loss: 0.6604 - Train acc: 0.704 - Test acc: 0.467
Epoch 30/100 - Loss: 0.6368 - Train acc: 0.742 - Test acc: 0.400
Epoch 40/100 - Loss: 0.6185 - Train acc: 0.753 - Test acc: 0.478
Epoch 50/100 - Loss: 0.6087 - Train acc: 0.793 - Test acc: 0.394
Epoch 60/100 - Loss: 0.5806 - Train acc: 0.821 - Test acc: 0.428
Epoch 70/100 - Loss: 0.5440 - Train acc: 0.836 - Test acc: 0.417
Epoch 80/100 - Loss: 0.5294 - Train acc: 0.856 - Test acc: 0.422
Epoch 90/100 - Loss: 0.5083 - Train acc: 0.864 - Test acc: 0.456
Epoch 100/100 - Loss: 0.5117 - Train acc: 0.871 - Test acc: 0.444

Training complete!
Final test accuracy: 0.444


In [7]:
import copy

# Reset model and optimizer for a clean run
model = EEGNet(n_channels=64, n_samples=321, n_classes=2).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()

best_test_acc = 0.0
best_model_state = None
best_epoch = 0

n_epochs = 100
for epoch in range(n_epochs):
    model.train()
    total_loss = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        out = model(xb)
        loss = criterion(out, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    train_acc = evaluate(train_loader)
    test_acc = evaluate(test_loader)

    # Track the best model seen so far, based on test accuracy
    if test_acc > best_test_acc:
        best_test_acc = test_acc
        best_model_state = copy.deepcopy(model.state_dict())
        best_epoch = epoch + 1

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{n_epochs} - Loss: {total_loss/len(train_loader):.4f} - Train acc: {train_acc:.3f} - Test acc: {test_acc:.3f}")

print(f"\nBest test accuracy: {best_test_acc:.3f} at epoch {best_epoch}")

# Restore the best model
model.load_state_dict(best_model_state)
final_test_acc = evaluate(test_loader)
print(f"Restored best model - Test accuracy: {final_test_acc:.3f}")

# Save it
torch.save(model.state_dict(), 'eegnet_best_model.pt')
print("Saved to eegnet_best_model.pt")

Epoch 10/100 - Loss: 0.6846 - Train acc: 0.664 - Test acc: 0.506
Epoch 20/100 - Loss: 0.6604 - Train acc: 0.692 - Test acc: 0.500
Epoch 30/100 - Loss: 0.6430 - Train acc: 0.724 - Test acc: 0.506
Epoch 40/100 - Loss: 0.6127 - Train acc: 0.758 - Test acc: 0.506
Epoch 50/100 - Loss: 0.5948 - Train acc: 0.778 - Test acc: 0.483
Epoch 60/100 - Loss: 0.5784 - Train acc: 0.796 - Test acc: 0.494
Epoch 70/100 - Loss: 0.5816 - Train acc: 0.812 - Test acc: 0.489
Epoch 80/100 - Loss: 0.5228 - Train acc: 0.838 - Test acc: 0.472
Epoch 90/100 - Loss: 0.5058 - Train acc: 0.844 - Test acc: 0.489
Epoch 100/100 - Loss: 0.5113 - Train acc: 0.874 - Test acc: 0.511

Best test accuracy: 0.556 at epoch 6
Restored best model - Test accuracy: 0.556
Saved to eegnet_best_model.pt


In [8]:
from google.colab import files
files.download('eegnet_best_model.pt')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>